In [ ]:
import os
import re
import subprocess
from glob import glob
from pathlib import Path

import numpy
import pandas as pd
from osgeo import gdal, gdalconst

import seabeepy as sb
from seabeepy.config import SETTINGS

In [ ]:
minio_client = sb.storage.minio_login(
    user=SETTINGS.MINIO_ACCESS_ID, password=SETTINGS.MINIO_SECRET_KEY
)

# Publish Marint Naturkart prediction maps

Rough notebook to publish habitat maps generated by NR for the Marint Naturkart project. For each level, the script publishes both a prediction map and a map of p-values indicating prediction confidence.

## 1. User input

In [ ]:
# Version of habitat annotation file used for training and prediction
version = "1-2"

# Levels to publish
levels = [1, 2]

# Missions to publish
mission_list = [
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/arendal_hovekilen_2025",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/karlsoy_2025/davoy",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/karlsoy_2025/grunnfjord",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/karlsoy_2025/langstranda",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/karlsoy_2025/lilleholmen",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/karlsoy_2025/steiness",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/karlsoy_2025/steinvollen",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/olbergholmen_msi/june_2021",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/olbergholmen_msi/june_2023",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/olbergholmen_msi/september_2023",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/remoy_2022_msi",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/runde_runde_2022_msi",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/vega_2022_msi/vega-n",
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/vega_2022_msi/vega-s",
]

# Working directory for temporary data
temp_dir = r"/home/notebook/cogs"

## 2. Publish datasets

In [ ]:
# Gets codes and names for annotation 'version'
df = sb.anno.get_class_codes(version)

# Publish selected levels
for level in levels:
    # Filter codes to level of interest
    code_len = level * 2
    df = df[df["code"].str.len() == code_len].drop_duplicates().reset_index()

    # Process mission data
    for mission_dir in mission_list:
        mission_name = sb.ortho.get_layer_name(mission_dir)
        for res_type in ["classifications", "pvalues"]:
            layer_name = f"{mission_name}_{res_type}_level{level}"
            print(f"\n################\nProcessing: {layer_name}")

            # Identify results
            results_dir = Path(mission_dir) / "results"
            if res_type == "classifications":
                regex = re.compile(rf"image_\d+_lev{level}\.tif$")
            else:
                regex = re.compile(rf"image_\d+_pvalue_lev{level}\.tif$")
            matches = [p for p in results_dir.glob("*.tif") if regex.match(p.name)]
            if len(matches) != 1:
                raise ValueError(f"Expected 1 match, found {len(matches)}: {matches}")
            res_path = str(matches[0])

            print("Standardising raster.")
            temp_path = os.path.join(temp_dir, f"{layer_name}.tif")
            stan_path = os.path.join(mission_dir, "results", f"{layer_name}.tif")
            nodata = sb.geo.get_geotiff_info(res_path)["nodata_value"]
            cmd = [
                "gdal_translate",
                "-of",
                "COG",
                "-co",
                "COMPRESS=LZW",
                "-co",
                "PREDICTOR=2",
                "-co",
                "NUM_THREADS=4",
                "-co",
                "OVERVIEWS=IGNORE_EXISTING",
                "-co",
                "BIGTIFF=YES",
                "-co",
                "OVERVIEW_RESAMPLING=NEAREST",
                "-co",
                "RESAMPLING=NEAREST",
                "-a_nodata",
                str(nodata),
                res_path,
                temp_path,
            ]
            subprocess.check_call(cmd)

            # Copy to MinIO and delete temporary file
            sb.storage.copy_file(temp_path, stan_path, minio_client, overwrite=True)
            os.remove(temp_path)

            print("Uploading to GeoServer.")
            if res_type == "classifications":
                sld_name = f"results_classes_v{version}_level{level}"
            else:
                sld_name = "pvalues"
            sb.geo.upload_raster_to_geoserver(
                stan_path,
                SETTINGS.GEOSERVER_USER,
                SETTINGS.GEOSERVER_PASSWORD,
                workspace="geonode",
                sld_name=sld_name,
            )

            print("Publishing to GeoNode.")
            sb.geo.publish_to_geonode(
                layer_name,
                SETTINGS.GEONODE_USER,
                SETTINGS.GEONODE_PASSWORD,
                workspace="geonode",
            )

            print("Updating metadata.")
            date = sb.ortho.parse_mission_data(mission_dir, parse_date=True)[2]
            abstract = f"Preliminary habitat {res_type} for '{mission_name}'."
            metadata = {
                "abstract": abstract,
                "date": date.isoformat(),
                "date_type": "creation",
                "attribution": "SeaBee",
            }
            sb.geo.update_geonode_metadata(
                layer_name,
                SETTINGS.GEONODE_USER,
                SETTINGS.GEONODE_PASSWORD,
                metadata,
            )